# Colab GPU 세션 준비

Colab 세션은 연결할 때마다 빈 머신에서 시작하므로, 새 세션마다 이 셀들을 위에서부터 실행합니다.

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 (VS Code에서는 Colab 커널 선택 시 GPU 런타임 선택)
2. 아래 `DATA_URL`에 데이터 zip의 Google Drive 공유 링크를 넣습니다. 공유 설정은 "링크가 있는 모든 사용자"여야 합니다.

In [16]:
BRANCH = "main"  # 작업 중인 브랜치를 쓰려면 브랜치 이름으로 바꾼다
DATA_URL = "https://drive.google.com/file/d/<파일ID>/view?usp=sharing"

## 1. GPU 확인

In [17]:
import torch

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 유형을 GPU로 바꾸세요."
print(torch.cuda.get_device_name(0))

Tesla T4


## 2. 저장소 받기

노트북은 VS Code(로컬)에서 열려 있어도 코드는 Colab 머신에서 실행되므로, 로컬의 프로젝트 파일을 import할 수 없습니다.
그래서 Colab 머신에 저장소를 받습니다. 같은 런타임이 살아 있으면 pull만 하고, 새 런타임이면 다시 clone합니다.
로컬에서 고친 코드는 push해야 Colab에 반영됩니다.

In [18]:
import importlib
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/pill-bbox")
if REPO_DIR.exists():
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/def-uk/pill-bbox.git {REPO_DIR}

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# 같은 커널에서 새로 받은 .py를 import할 수 있도록 import 캐시를 비운다
importlib.invalidate_caches()
!git branch --show-current && git log --oneline -1

06f1dc4 (HEAD -> main, origin/main, origin/HEAD) Chore/colab score (#32)


## 3. 패키지 설치 (uv.lock 기준)

Colab 커널은 `.venv`가 아니라 Colab의 Python으로 돌아가므로, `uv sync` 대신 `uv.lock`에 고정된 버전을 커널의 Python에 그대로 설치합니다.
팀원 로컬 환경과 같은 버전이 깔립니다. torch는 lock에 적힌 CUDA 12.6 빌드로 바뀝니다.

In [19]:
if not Path("~/.local/bin/uv").expanduser().exists():
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{Path('~/.local/bin').expanduser()}:{os.environ['PATH']}"

!uv export -q --frozen --no-dev --no-hashes --no-emit-project -o /tmp/requirements.txt
!uv pip install --python {sys.executable} -r /tmp/requirements.txt --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match

downloading uv 0.13.5 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.13.16 environment at: /usr
Checked 106 packages in 14ms


## 4. 데이터 받기

`data/raw`에 이미 파일이 있으면 건너뜁니다. 다시 받으려면 `overwrite=True`.

In [20]:
from data_process.download_data import download_data

download_data(DATA_URL, Path("data/raw"))
!ls data/raw

ModuleNotFoundError: No module named 'data_process.download_data'

## 5. wandb 로그인 (선택)

Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `WANDB_API_KEY`를 등록하고 노트북 액세스를 켭니다. 등록하지 않으면 wandb 없이 진행합니다.

In [ ]:
try:
    from google.colab import userdata

    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY가 없어 wandb를 끄고 진행합니다.")

## 6. 확인

In [ ]:
from src.config import get_device

print(get_device())  # cuda가 나와야 한다